# Poses vs Ligands — Phase 2 UX evaluation (DDOS-6736)

This notebook compares **legacy pose-hydrated `Ligand` objects** with the new first-class **`Pose` / `PoseSet`** types.

Run against the local mock server or a dev org after `deeporigin login`.

**Phase 2 scope:** additive API only. `Docking.get_poses()` still returns `LigandSet` until DDOS-6737.

In [ ]:
from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    Docking,
    Ligand,
    LigandSet,
    Pose,
    PoseSet,
    Pocket,
    Protein,
)
from deeporigin.platform.client import DeepOriginClient

client = DeepOriginClient()
print(client)

## 1. Before vs after: ID clarity

Legacy docking returns `LigandSet` where the **pose result id** hides in `properties["id"]` while `Ligand.id` is the **ligand table id**.

New `PoseSet` exposes `pose.id` (pose result) and `pose.ligand_id` (parent ligand) explicitly.

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.remove_water()
protein.upload(client=client, remote_path="testing/brd.pdb")

pocket = Pocket.from_pdb_file(BRD_DATA_DIR / "pocket.pdb")
ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
ligand.sync(client=client, remote_path="testing/brd-2.sdf")

docking = Docking(protein=protein, pocket=pocket, ligand=ligand, client=client)
legacy_poses: LigandSet = docking.run()

print("Legacy LigandSet (first pose):")
legacy = legacy_poses[0]
print("  Ligand.id (ligand table):     ", legacy.id)
print("  properties['id'] (pose row):  ", legacy.properties.get("id"))

pose_set = PoseSet.from_result(execution_id=docking.id, client=client)
print("\nNew PoseSet (first pose):")
pose = pose_set[0]
print("  Pose.ligand_id:               ", pose.ligand_id)
print("  Pose.id (pose result):        ", pose.id)
print("  IDs distinct:", pose.id != pose.ligand_id)

## 2. Parent → child discovery: `Ligand.get_poses()`

In [ ]:
children = ligand.get_poses(client=client)
print(f"Ligand {ligand.id} has {len(children)} pose(s)")
for p in children:
    print(f"  pose.id={p.id}  score={p.pose_score}  origin={p.origin}")

## 3. Register an external SDF: `Pose.from_sdf()`

Uses ImportTool pose registration (mock server locally; real tool when DDOS-6734 lands).

In [ ]:
registered = Pose.from_sdf(
    BRD_DATA_DIR / "brd-3.sdf",
    protein_id=protein.id,
    client=client,
)
print(registered)
print("Registered pose id:", registered.id)
print("Parent ligand id:", registered.ligand_id)

## 4. What is still legacy (Phase 3 — DDOS-6737)

`Docking.get_poses()` continues to return `LigandSet` in Phase 2. Compare:

In [ ]:
sdf_poses = docking.get_poses()
print(type(sdf_poses), "← still LigandSet until breaking change")
print("Migration path: PoseSet.from_result(execution_id=docking.id)")